##This notebook is a stepping process through the validation processes involved with truth detection. This includes things like library importing, EDA, model training, API configuration, etc.

# Exploratory Walkthrough of Statistical Truth Detection
This serves to decide the 

## Step 0: Imports

In [2]:
# Upgrade pip and setuptools
!pip install --upgrade pip setuptools wheel

# core and NLP libraries
!pip install numpy pandas matplotlib scikit-learn nltk textblob spacy requests torch

# get teh fever datasets and pyarrow bc error with version control
!pip install \
    transformers==4.33.3 \
    keras==2.11.0 \
    tensorflow==2.11.0 \
    datasets==2.13.1 \
    fsspec==2023.6.0 \
    pyarrow==15.0.2 \
    huggingface_hub==0.17.3 \
    git+https://github.com/UKPLab/sentence-transformers.git@v2.2.2

# get the rag libraries
!pip install llama-index sentence-transformers faiss-cpu wikipedia 

# tried to include these, but seems to have broken the others
# mteb llama-index-vector-stores-faiss llama-index-embeddings-openai llama-index-embeddings-huggingface

# get the spacy English model
# gives URL error: !python -m spacy download en_core_web_sm
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 119.8 kB/s eta 0:00:00 kB/s eta 0:00:0211
  Attempting uninstall: wheel
    Found existing installation: wheel 0.44.0
    Uninstalling wheel-0.44.0:
      Successfully uninstalled wheel-0.44.0
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.3/624.3 kB 1.7 MB/s  0:00:00m ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 380.5 kB/s  0:00:153.1 kB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 636.7/636.7 kB 433.5 kB/s  0:00:01.3 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.2/890.2 kB 382.9 kB/s  0:00:02.6 kB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 352.6 kB/s  0:00:18a 0:00

In [1]:
# imports
import importlib
import re
import requests
import spacy
import json
import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
import numpy as np
from typing import List, Set
# for NLP
import nltk
# for tokenizing, using pretrained model (BERT, literature)
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
# sentence detection/classification
from textblob import TextBlob
# for RAG: https://developers.llamaindex.ai/python/framework/#introduction
import llama_index 
# for FEVER
import datasets
import pyarrow
# from datasets import load_dataset
# models/classification
import torch
import faiss
import wikipedia
import wikipediaapi
# import mteb

# get FEVER
# this dataset as starting point but likely use the full json: https://huggingface.co/datasets?modality=modality:text&sort=trending&search=fever
# full json: https://fever.ai/dataset/fever.html

# get NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('brown')

# download the scapy english model
nlp = spacy.load("en_core_web_sm")

# autoupdate 
%load_ext autoreload
%autoreload 2

print("All installations complete!")

2025-10-27 19:15:19.897127: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
[nltk_data] Downloading package punkt to /Users/admin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/admin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /Users/admin/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package brown to /Users/admin/nltk_data...
[nltk_data]   Package brown is already up-to-date!
/Users/admin/Documents/graduate/penn/courses/dats598/backend/.venv/lib/python3.10/site-packages/coreferee

All installations complete!


## Step 1: Load FEVER

In [2]:
from datasets import load_dataset

fever_configs = ['v1.0', 'v2.0', 'wiki_pages']
fever = load_dataset("fever", fever_configs[0], cache_dir="./data/hf_cache")
fev_train = fever["train"].to_pandas()
fev_train.head()

Using the latest cached version of the dataset since fever couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'v1.0' at data/hf_cache/fever/v1.0/1.0.0/7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e (last modified on Wed Oct 22 12:58:46 2025).


,id,label,claim,evidence_annotation_id,evidence_id,evidence_wiki_url,evidence_sentence_id
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Nikolaj_Coster-Waldau,7
1,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,92206,104971,Fox_Broadcasting_Company,-1
2,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187498,Roman_Atwood,1
3,150448,SUPPORTS,Roman Atwood is a content creator.,174271,187499,Roman_Atwood,3
4,214861,SUPPORTS,"History of art includes architecture, dance, s...",255136,254645,History_of_art,2


## Step 2: Claim Detection from Text

In [3]:
import test_claim_detection, test_claim_extraction
# test_claim_detection.main(n_claims=1000)
test_claim_extraction.main(n_claims=1000)

Using the latest cached version of the dataset since fever couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'v1.0' at data/hf_cache/fever/v1.0/1.0.0/7f8936e0558704771b08c7ce9cc202071b29a0050603374507ba61d23c00a58e (last modified on Wed Oct 22 12:58:46 2025).


Average semantic similarity: 0.734


## Step 3: ER

In [4]:
import wikipedia
from sentence_transformers import SentenceTransformer
from nltk import sent_tokenize

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Fetch Wikipedia content
topic = "Artificial intelligence"
text = wikipedia.page(topic).content

# Chunk into sentences (or paragraphs)
chunks = sent_tokenize(text) 
chunks = [c.strip() for c in chunks if len(c.strip()) > 50]

# Embed the chunks
embeddings = model.encode(chunks, convert_to_numpy=True)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 4a5e3093-b65a-47c3-a1d6-b2bc245ee18d)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 02fd650c-2d2f-43a3-be15-7c146d5e7b9c)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./README.md
Retrying in 2s [Retry 2/5].


KeyboardInterrupt: 

In [53]:
import wikipediaapi
from claim_pipe import ClaimDetectionPipeline as cdp
from wiki_evidence_retrieval_pipeline import WikiEvidenceRetrievalPipeline as erp

claim_pipe = cdp()
evidence_pipe = erp()

text = """
The Eiffel Tower was built in 1889 for the World's Fair.
It stands 324 meters tall and is located in Paris, France.
"""

claims = claim_pipe.process_text(text)
evidence_results = evidence_pipe.batch_retrieve_evidence(claims)
for claim in claims:
    print(f"\nClaim: {claim['text']}")
    print(f"Confidence: {claim['confidence']:.2f}")
    
    evidence_list = evidence_results[claim['text']]
    print(f"Found {len(evidence_list)} pieces of evidence:")
    
    for i, evidence in enumerate(evidence_list[:3], 1):
        print(f" evidence source: {evidence['source_title']}")
        print(f" relevance score: {evidence['relevance_score']:.2f}")
        print(f" data: {evidence['text'][:150]}...")

Wikipedia search error: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=opensearch&search=It+324+meters+Paris&limit=3&format=json
Wikipedia search error: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /w/api.php?action=opensearch&search=Paris+France&limit=3&format=json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x18808bf40>, 'Connection to en.wikipedia.org timed out. (connect timeout=5)'))

Claim: It stands 324 meters tall and is located in Paris, France.
Confidence: 0.60


In [ ]:
# from llama_index.vector_stores.faiss import FaissVectorStore
# from llama_index.embeddings.openai import OpenAIEmbedding
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, load_index_from_storage, StorageContext
# from llama_index.core.query_engine import RetrieverQueryEngine


import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

wiki_dir = "../data/wikipedia/extracted"
# read the docs
# documents = SimpleDirectoryReader(wiki_dir).load_data()
# initialize embedding model and FAISS index
# trade off on embedding models
openai_models = ["text-embedding-3-small","text-embedding-3-large"]
hf_models = ["sentence-transformers/all-MiniLM-L6-v2","BAAI/bge-base-en"]
# embed_model = OpenAIEmbedding(model="text-embedding-3-small")
# embed_model = HuggingFaceEmbedding(model_name=hf_models[0])
# HuggingFaceEmbedding(model_name=embed_model)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
# embed_model = mteb.get_model(model_name)
# faiss_index = faiss.IndexFlatL2(embed_model.dimensions)
# vector_store = FaissVectorStore(faiss_index=faiss_index)
# # build and persist index
# storage_context = StorageContext.from_defaults(vector_store=vector_store)
# index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

# index.storage_context.persist(persist_dir="data/wiki_faiss_index")

# # load the index
# storage_context = StorageContext.from_defaults(persist_dir="data/wiki_faiss_index")
# index = load_index_from_storage(storage_context)
# query_engine = index.as_query_engine()
# response = query_engine.query("When was the Eiffel Tower built?")
# print(response)

documents = []
filenames = []

for root, dirs, files in os.walk(wiki_dir):
    for fname in files:
        if fname.startswith('wiki'):
            full_path = os.path.join(root, fname)
            try:
                with open(full_path, "r", encoding="utf-8") as f:
                    text = f.read().strip()
                    if text:  # skip empty files
                        documents.append(text)
                        filenames.append(full_path)
            except Exception as e:
                print(f"Could not read {full_path}: {e}")

print(f"Loaded {len(documents)} text files from {wiki_dir}")


model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)


# print("Embedding documents...")
embeddings = embedder.encode(documents, show_progress_bar=True, convert_to_numpy=True)

# # ----------------------------
# # 4. Build FAISS index
# # ----------------------------
embedding_dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dim)
faiss_index.add(embeddings)

print(f"FAISS index built with {faiss_index.ntotal} vectors.")

['.DS_Store']
['wiki_73', 'wiki_87', 'wiki_80', 'wiki_74', 'wiki_89', 'wiki_42', 'wiki_45', 'wiki_11', 'wiki_16', 'wiki_29', 'wiki_20', 'wiki_27', 'wiki_18', 'wiki_44', 'wiki_88', 'wiki_43', 'wiki_75', 'wiki_81', 'wiki_86', 'wiki_72', 'wiki_26', 'wiki_19', 'wiki_21', 'wiki_17', 'wiki_28', 'wiki_10', 'wiki_32', 'wiki_35', 'wiki_03', 'wiki_04', 'wiki_50', 'wiki_57', 'wiki_68', 'wiki_61', 'wiki_95', 'wiki_92', 'wiki_66', 'wiki_59', 'wiki_05', 'wiki_02', 'wiki_34', 'wiki_33', 'wiki_67', 'wiki_93', 'wiki_58', 'wiki_94', 'wiki_60', 'wiki_56', 'wiki_69', 'wiki_51', 'wiki_15', 'wiki_12', 'wiki_24', 'wiki_23', 'wiki_48', 'wiki_83', 'wiki_77', 'wiki_70', 'wiki_84', 'wiki_79', 'wiki_46', 'wiki_41', 'wiki_22', 'wiki_25', 'wiki_13', 'wiki_14', 'wiki_40', 'wiki_78', 'wiki_47', 'wiki_85', 'wiki_71', 'wiki_49', 'wiki_76', 'wiki_82', 'wiki_54', 'wiki_53', 'wiki_98', 'wiki_91', 'wiki_65', 'wiki_62', 'wiki_96', 'wiki_09', 'wiki_36', 'wiki_31', 'wiki_38', 'wiki_07', 'wiki_00', 'wiki_97', 'wiki_63', 'wiki_

Batches:   0%|          | 0/613 [00:00<?, ?it/s]

In [ ]:
def search(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = faiss_index.search(query_vec, top_k)
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"\nResult {rank+1}:")
        print(f"file: {filenames[idx]}")
        print(f"distance: {dist:.4f}")
        print("text:", documents[idx][:400].replace("\n", " ") + "...")
    return indices, distances
    
search("Eiffel Tower built 1765")

In [ ]:
import pickle
save_dir = "data/wiki_faiss_vdb"
os.makedirs(save_dir, exist_ok=True)
faiss.write_index(faiss_index, os.path.join(save_dir, "index.faiss"))
with open(os.path.join(save_dir, "metadata.pkl"), "wb") as f:
    pickle.dump({"filenames": filenames, "documents": documents}, f)